# Sprint 1 — Cognitive RAG Experiment

**Goal**: Test whether templates help the LLM correctly **retrieve and ground** answers from docs.

**Primary metrics** (RAG-focused): Evidence markers, faithfulness, doc-level retrieval.
**Secondary**: Overall quality % (heuristic evaluator).

| Condition | Prompt Type | Knowledge | LLM Calls |
|---|---|---|---|
| **A. Raw** | None | None | 1 |
| **B. RAG only** | None | Injected docs | 1 |
| **C. Generic prompt only** | DiagnosticRootCauseAnalyzer.generic_prompt | None | 1 |
| **D. Generic prompt + RAG** | DiagnosticRootCauseAnalyzer.generic_prompt + symptoms | Injected docs | 1 |
| **E. RagAnswerer** | RagAnswerer (RAG template) | Injected docs | 1 |

We simulate RAG by providing realistic "retrieved documents" as text. In production, these would come from a vector store. **E** tests the dedicated RAG template.

---

## Prerequisites

1. Have an **OpenAI API key** set: `export OPENAI_API_KEY=sk-...`
2. Run from the repo root so `mycontext` is importable
3. `pip install mycontext-ai litellm`
4. **DiagnosticRootCauseAnalyzer** is an enterprise template — ensure your environment has enterprise access if license checks apply

In [ ]:
!pip install litellm

In [1]:
import os
import time

os.environ.setdefault('OPENAI_API_KEY', 'sk-...')  # Set your key here or via environment variable
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
os.environ['GOOGLE_API_KEY'] = 'AIza...'

PROVIDER = 'openai'

In [2]:
from mycontext.core import Context
from mycontext.foundation import Directive, Guidance
from mycontext.intelligence.output_evaluator import OutputDimension, OutputEvaluator

evaluator = OutputEvaluator(mode='heuristic')
print('Components loaded.')

Components loaded.


## Step 1 — Simulated Retrieved Documents

These represent what a RAG pipeline would retrieve from a company's knowledge base. We provide realistic business data so we can measure whether the template helps the LLM **reason** over it, not just summarize.

In [3]:
QUESTION = "Why did customer churn spike 40% last quarter and what should we do about it?"

RETRIEVED_DOCS = """## Retrieved Document 1: Customer Support Ticket Analysis (Q4 2025)
- Total tickets: 4,832 (up 62% from Q3)
- Top category: "Billing errors" (28% of tickets, up from 8%)
- Second: "Feature missing after update" (19%)
- Third: "Slow performance" (15%)
- Average resolution time: 4.2 days (was 1.8 days in Q3)
- CSAT score dropped from 4.1 to 2.9

## Retrieved Document 2: Product Changelog (Q4 2025)
- Oct 3: Migrated billing system from Stripe v2 to Stripe v4 (internal mandate)
- Oct 17: Launched "Simplified UI" — removed 12 features deemed low-usage
- Nov 8: Pricing increase announced: Pro plan $49 → $79/mo (effective Jan 1)
- Nov 22: Performance hotfix for dashboard (resolved 60% of slow queries)
- Dec 5: Re-added 4 of 12 removed features after customer backlash

## Retrieved Document 3: Churn Exit Survey (134 responses, Q4 2025)
- 41% cited "features I relied on were removed"
- 29% cited "billing issues / overcharges"
- 18% cited "too expensive now"
- 12% cited "found a better alternative"
- Verbatim: "I was charged 3x my normal amount after the billing migration"
- Verbatim: "The simplified UI removed my most-used workflow"

## Retrieved Document 4: Revenue & Retention Metrics
- Q3 monthly churn: 3.2%
- Q4 monthly churn: 5.5% (Oct: 4.1%, Nov: 5.8%, Dec: 6.6%)
- Q4 MRR lost to churn: $184K (Q3: $112K)
- Net Revenue Retention: 89% (was 104% in Q3)
- Expansion revenue: flat at $43K (existing customers stopped upgrading)
"""

print(f'Question: {QUESTION}')
print(f'Retrieved docs: {len(RETRIEVED_DOCS)} chars, {len(RETRIEVED_DOCS.split())} words')

Question: Why did customer churn spike 40% last quarter and what should we do about it?
Retrieved docs: 1441 chars, 246 words


In [4]:
import litellm
litellm.drop_params = True
# DiagnosticRootCauseAnalyzer (enterprise) used in Step 2 — requires enterprise license if applicable

In [5]:
from mycontext.templates.free.reasoning import RootCauseAnalyzer

rca = RootCauseAnalyzer()
prompt = rca.generic_prompt(
    problem="API latency spiked to 8 seconds",
    depth="thorough",
)
print(prompt)

You are a root cause analysis specialist and systems thinker. Investigate the following problem to identify its true root cause:

Problem: API latency spiked to 8 seconds

Analysis depth: thorough

Apply a rigorous diagnostic methodology: (1) Define the problem precisely, separating symptoms from underlying causes. (2) Conduct a Five Whys analysis — ask 'why' iteratively to drill beneath surface-level explanations. (3) Perform an Ishikawa (fishbone) analysis examining causes across people, process, technology, environment, management, and materials. (4) Identify contributing factors and their interactions. (5) Verify each candidate cause with available evidence. (6) Assess systemic factors and feedback loops. (7) State the root cause clearly and explain the causal chain from root to symptoms. (8) Recommend specific prevention strategies and lessons learned.

Distinguish symptoms from causes throughout. Be systematic and evidence-driven.


## Step 2 — Run All 4 Conditions

In [6]:
from mycontext.templates.enterprise.diagnostic import DiagnosticRootCauseAnalyzer

conditions = {}
analyzer = DiagnosticRootCauseAnalyzer()

# --- Condition A: Raw (just the question) ---
print('[A] Raw prompt...')
ctx_a = Context(directive=Directive(content=QUESTION))
t0 = time.time()
res_a = ctx_a.execute(provider=PROVIDER)
conditions['A_raw'] = {
    'ctx': ctx_a,
    'output': res_a.response,
    'time': time.time() - t0,
}
print(f'  Done in {conditions["A_raw"]["time"]:.1f}s')

# --- Condition B: RAG only (docs stuffed into context, no template) ---
print('[B] RAG only (docs + question, no template)...')
ctx_b = Context(
    guidance=Guidance(role='Business analyst'),
    directive=Directive(content=QUESTION),
    knowledge=RETRIEVED_DOCS,
)
t0 = time.time()
res_b = ctx_b.execute(provider=PROVIDER)
conditions['B_rag_only'] = {
    'ctx': ctx_b,
    'output': res_b.response,
    'time': time.time() - t0,
}
print(f'  Done in {conditions["B_rag_only"]["time"]:.1f}s')

# --- Condition C: Generic DiagnosticRCA prompt only (no docs) ---
print('[C] Generic prompt only (DiagnosticRootCauseAnalyzer.generic_prompt, no docs)...')
prompt_c = analyzer.generic_prompt(problem=QUESTION, symptoms="")
ctx_c = Context(directive=Directive(content=prompt_c))
t0 = time.time()
res_c = ctx_c.execute(provider=PROVIDER)
conditions['C_template_only'] = {
    'ctx': ctx_c,
    'output': res_c.response,
    'time': time.time() - t0,
}
print(f'  Done in {conditions["C_template_only"]["time"]:.1f}s')

# --- Condition D: Generic DiagnosticRCA prompt + RAG docs (symptoms = retrieved data) ---
print('[D] Generic prompt + retrieved docs...')
prompt_d = analyzer.generic_prompt(problem=QUESTION, symptoms=RETRIEVED_DOCS)
ctx_d = Context(directive=Directive(content=prompt_d))
t0 = time.time()
res_d = ctx_d.execute(provider=PROVIDER)
conditions['D_cognitive_rag'] = {
    'ctx': ctx_d,
    'output': res_d.response,
    'time': time.time() - t0,
}
print(f'  Done in {conditions["D_cognitive_rag"]["time"]:.1f}s')

# --- Condition E: RagAnswerer (dedicated RAG template) ---
from mycontext.templates.free.specialized import RagAnswerer
print('[E] RagAnswerer (RAG template)...')
rag = RagAnswerer()
ctx_e = rag.build_context(question=QUESTION, retrieved_docs=RETRIEVED_DOCS, task="answer")
t0 = time.time()
res_e = ctx_e.execute(provider=PROVIDER)
conditions['E_rag_answerer'] = {
    'ctx': ctx_e,
    'output': res_e.response,
    'time': time.time() - t0,
}
print(f'  Done in {conditions["E_rag_answerer"]["time"]:.1f}s')

print('\nAll 5 conditions complete!')

[A] Raw prompt...
  Done in 12.1s
[B] RAG only (docs + question, no template)...
  Done in 14.5s
[C] Generic prompt only (DiagnosticRootCauseAnalyzer.generic_prompt, no docs)...
  Done in 18.6s
[D] Generic prompt + retrieved docs...
  Done in 25.5s
[E] RagAnswerer (RAG template)...
  Done in 5.3s

All 5 conditions complete!


## Step 3 — Quality Scores (Secondary)

Heuristic evaluator. **Primary RAG metrics** (evidence markers, faithfulness, doc retrieval) are in Step 5 below.

In [7]:
print(f'{"Condition":<25} {"Overall":>9} {"Instruct":>9} {"Reason":>9} {"Action":>9} {"Structure":>10} {"Scaffold":>10} {"Time":>7}')
print('-' * 98)

scores = {}
labels = {
    'A_raw': 'A. Raw prompt',
    'B_rag_only': 'B. RAG only',
    'C_template_only': 'C. Generic DiagnosticRCA only',
    'D_cognitive_rag': 'D. Generic DiagnosticRCA + RAG',
    'E_rag_answerer': 'E. RagAnswerer',
}

for key, label in labels.items():
    data = conditions[key]
    score = evaluator.evaluate(data['ctx'], data['output'])
    scores[key] = score
    dims = score.dimensions
    print(
        f'{label:<25} '
        f'{score.overall:>8.1%} '
        f'{dims[OutputDimension.INSTRUCTION_FOLLOWING]:>8.1%} '
        f'{dims[OutputDimension.REASONING_DEPTH]:>8.1%} '
        f'{dims[OutputDimension.ACTIONABILITY]:>8.1%} '
        f'{dims[OutputDimension.STRUCTURE_COMPLIANCE]:>9.1%} '
        f'{dims[OutputDimension.COGNITIVE_SCAFFOLDING]:>9.1%} '
        f'{data["time"]:>6.1f}s'
    )

print()
# Compute lift of Cognitive RAG over each other condition
d_score = scores['D_cognitive_rag'].overall
for key in ['A_raw', 'B_rag_only', 'C_template_only', 'E_rag_answerer']:
    other = scores[key].overall
    lift = ((d_score / max(other, 0.01)) - 1) * 100
    print(f'D (Generic+RAG) vs {labels[key]}: {lift:+.1f}% lift')

Condition                   Overall  Instruct    Reason    Action  Structure   Scaffold    Time
--------------------------------------------------------------------------------------------------
A. Raw prompt                66.3%    50.0%    62.6%   100.0%     75.0%     50.0%   12.1s
B. RAG only                  90.1%   100.0%    73.1%   100.0%     70.0%    100.0%   14.5s
C. Generic DiagnosticRCA only    96.2%   100.0%   100.0%   100.0%     75.0%    100.0%   18.6s
D. Generic DiagnosticRCA + RAG    95.5%   100.0%   100.0%   100.0%     70.0%    100.0%   25.5s
E. RagAnswerer               47.7%    40.0%    40.9%    65.0%     30.0%     60.0%    5.3s

D (Generic+RAG) vs A. Raw prompt: +44.1% lift
D (Generic+RAG) vs B. RAG only: +6.0% lift
D (Generic+RAG) vs C. Generic DiagnosticRCA only: -0.8% lift
D (Generic+RAG) vs E. RagAnswerer: +100.3% lift


## Step 4 — Read the Outputs

Compare what the LLM actually produced in each condition.

In [8]:
for key, label in labels.items():
    data = conditions[key]
    score = scores[key]
    print(f'\n{"="*70}')
    print(f'{label} — Overall: {score.overall:.1%}')
    print(f'Strengths: {", ".join(score.strengths[:3])}')
    print(f'Weaknesses: {", ".join(score.weaknesses[:3])}')
    print('='*70)
    print(data['output'][:2000])
    if len(data['output']) > 2000:
        print(f'\n... ({len(data["output"])} total chars)')


A. Raw prompt — Overall: 66.3%
Strengths: Strong Actionability, Strong Structure Compliance
Weaknesses: 
To determine why customer churn spiked by 40% last quarter, we should consider several potential factors:

1. **Customer Feedback**: Analyze feedback from customers who canceled their subscriptions. Look for common themes or issues, such as dissatisfaction with product features, pricing, customer service, or usability.

2. **Market Trends**: Investigate any recent shifts in the market or industry that could have influenced customer behavior. This could include new competitors entering the market, changes in consumer preferences, or economic factors affecting spending.

3. **Product Changes**: Assess if there were any changes to your product or service that could have negatively impacted customer satisfaction. This could include updates that introduced bugs, removed popular features, or increased prices.

4. **Customer Engagement**: Review customer engagement metrics. A decline in u

## Step 5 — RAG Metrics (Primary)

Does the output correctly retrieve facts from the docs?
- **3a Evidence markers**: How many specific facts appear?
- **3b Faithfulness**: Numeric claims not in docs = possible hallucination
- **3c Doc-level retrieval**: Facts retrieved per source document

In [9]:
# Check for specific data points from the retrieved docs
# Each entry: (display_name, [alternates]) — matches if ANY alternate appears (LLMs often paraphrase)
EVIDENCE_MARKERS = [
    ('billing', ['billing']),
    ('Stripe', ['stripe']),
    ('4,832', ['4,832', '4832', '4 832']),
    ('28%', ['28%', '28 percent']),
    ('41%', ['41%', '41 percent']),
    ('$79', ['$79', '79/mo', '79 per month']),
    ('CSAT', ['csat', 'customer satisfaction']),
    ('2.9', ['2.9']),
    ('12 features', ['12 features', 'twelve features']),
    ('5.5%', ['5.5%', '5.5 percent']),
    ('$184K', ['$184k', '184k', '184,000', '$184,000']),
    ('89%', ['89%', '89 percent', 'eighty-nine']),
    ('exit survey', ['exit survey']),
    ('Oct', ['oct', 'october']),
    ('Nov', ['nov', 'november']),
    ('Dec', ['dec', 'december']),
]

print(f'{"Evidence marker":<20} {"A_raw":>8} {"B_rag":>8} {"C_tmpl":>8} {"D_cog":>8} {"E_rag":>8}')
print('-' * 64)

totals = dict.fromkeys(labels, 0)
for display_name, alternates in EVIDENCE_MARKERS:
    row = {}
    for key in labels:
        out_lower = conditions[key]['output'].lower()
        found = any(alt.lower() in out_lower for alt in alternates)
        row[key] = 'YES' if found else '-'
        if found:
            totals[key] += 1
    print(f'{display_name:<20} {row["A_raw"]:>8} {row["B_rag_only"]:>8} {row["C_template_only"]:>8} {row["D_cognitive_rag"]:>8} {row["E_rag_answerer"]:>8}')

print('-' * 64)
print(f'{"TOTAL":>20} {totals["A_raw"]:>8} {totals["B_rag_only"]:>8} {totals["C_template_only"]:>8} {totals["D_cognitive_rag"]:>8} {totals["E_rag_answerer"]:>8}')
print(f'\nMax possible: {len(EVIDENCE_MARKERS)}')
print('\nCondition D or E should have highest count — D: DiagnosticRCA + docs; E: RagAnswerer.')

# --- 3b: Faithfulness check — flag numbers/percentages in output not in docs ---
import re

def extract_numeric_claims(text):
    """Extract numbers, percentages, dollar amounts that could be factual claims."""
    patterns = [
        r'\d{1,3}(?:,\d{3})*(?:\.\d+)?%',  # 28%, 5.5%, 41%
        r'\$[\d,]+(?:K|M)?',  # $79, $184K
        r'\b\d{1,3}(?:,\d{3})+\b',  # 4,832
        r'\b\d+\.\d+\b',  # 2.9, 4.2
    ]
    found = set()
    for p in patterns:
        for m in re.finditer(p, text, re.IGNORECASE):
            found.add(m.group())
    return found

docs_numerics = extract_numeric_claims(RETRIEVED_DOCS)
print('\n--- Faithfulness Check (numeric claims not in docs = possible hallucination) ---')
print(f'{"Condition":<25} {"In docs":>8} {"Not in docs":>12} {"Flagged":>10}')
print('-' * 58)
faithfulness = {}
for key in labels:
    out_numerics = extract_numeric_claims(conditions[key]['output'])
    in_docs = out_numerics & docs_numerics
    not_in_docs = out_numerics - docs_numerics
    faithfulness[key] = {'in_docs': len(in_docs), 'not_in_docs': len(not_in_docs), 'flagged': list(not_in_docs)[:5]}
    flagged_str = ', '.join(list(not_in_docs)[:3]) if not_in_docs else '-'
    if len(not_in_docs) > 3:
        flagged_str += f' (+{len(not_in_docs)-3})'
    print(f'{labels[key]:<25} {len(in_docs):>8} {len(not_in_docs):>12} {str(flagged_str)[:25]}')

# --- 3c: Doc-level retrieval (which doc's facts were retrieved?) ---
DOC_MARKERS = {
    'Doc1_Support': ['4,832', '28%', 'billing', 'CSAT', '2.9'],
    'Doc2_Changelog': ['Stripe', '12 features', '$79', 'Oct', 'Nov', 'Dec'],
    'Doc3_Survey': ['41%', 'exit survey', '29%', '18%'],
    'Doc4_Revenue': ['5.5%', '$184K', '89%'],
}
print('\n--- Doc-Level Retrieval (facts per source doc) ---')
print(f'{"Condition":<25} {"Doc1":>6} {"Doc2":>6} {"Doc3":>6} {"Doc4":>6} {"Total":>6}')
print('-' * 58)
doc_retrieval = {}
for key in labels:
    out_lower = conditions[key]['output'].lower()
    row = {}
    for doc_name, markers in DOC_MARKERS.items():
        count = sum(1 for m in markers if m.lower() in out_lower)
        row[doc_name] = count
    doc_retrieval[key] = row
    total = sum(row.values())
    print(f'{labels[key]:<25} {row["Doc1_Support"]:>6} {row["Doc2_Changelog"]:>6} {row["Doc3_Survey"]:>6} {row["Doc4_Revenue"]:>6} {total:>6}')


Evidence marker         A_raw    B_rag   C_tmpl    D_cog    E_rag
----------------------------------------------------------------
billing                     -      YES        -      YES      YES
Stripe                      -      YES        -      YES        -
4,832                       -        -        -      YES        -
28%                         -      YES        -      YES      YES
41%                         -      YES        -      YES      YES
$79                         -      YES        -        -        -
CSAT                      YES      YES      YES      YES        -
2.9                         -      YES        -      YES        -
12 features                 -      YES        -        -      YES
5.5%                        -      YES        -      YES      YES
$184K                       -        -        -      YES        -
89%                         -        -        -      YES        -
exit survey               YES      YES        -        -        -
Oct        

## Step 6 — Save Results

In [ ]:
import json
from datetime import datetime

report = {
    'sprint': 'Sprint 1 — Cognitive RAG Experiment',
    'date': datetime.now().isoformat(),
    'provider': PROVIDER,
    'question': QUESTION,
    'retrieved_docs_chars': len(RETRIEVED_DOCS),
    'conditions': {},
}

for key, label in labels.items():
    s = scores[key]
    report['conditions'][key] = {
        'label': label,
        'overall': round(s.overall, 4),
        'dimensions': {d.value: round(v, 4) for d, v in s.dimensions.items()},
        'time_seconds': round(conditions[key]['time'], 2),
        'output_length': len(conditions[key]['output']),
        'evidence_markers_found': totals[key],
        'faithfulness': faithfulness.get(key, {}),
        'doc_retrieval': doc_retrieval.get(key, {}),
    }

outpath = 'sprint1_cognitive_rag_results.json'
with open(outpath, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Results saved to {outpath}')